# 02 — SAE feature browsing

**Phase 2** of `vault/01-projects/code/mech-interp-codebase.md`: load a Gemma Scope SAE for a chosen residual-stream layer, encode Phase-1 cached activations, identify the top-k features that fire *differentially* on the refusal vs compliance contrast pair, pull Neuronpedia autointerp labels for each, and write a one-paragraph human gloss.

## What this notebook does

1. Re-caches Gemma-2-2B residual-stream activations using `hook_resid_post` at a single mid-to-late layer (default: layer 20) — Phase 1 cached `hook_resid_pre`, but Gemma Scope's residual-stream SAEs are trained on `resid_post` and Neuronpedia hosts the dashboards under that name.
2. Loads the canonical width-16k Gemma Scope SAE for that layer via SAELens.
3. Encodes both prompts; produces per-feature activation vectors and a *differential* score `mean(|f_refusal|) − mean(|f_compliance|)` over feature dimensions.
4. Plots the top-k differential features in both directions (refusal-leaning and compliance-leaning), with their feature indices.
5. Pulls Neuronpedia autointerp descriptions for the top-k features (best-effort; degrades to per-feature stubs if the API call fails, so the notebook still runs offline).
6. Asks the human (you) to write a one-paragraph gloss per feature into the final cell.
7. Checkpoints the per-feature activations to `../data/phase2_sae_features.pt` for Phase 3 attribution-graph annotation.

## Why this matters

Elhage et al. 2022's *Toy Models of Superposition* gave the mathematical motivation: features in a transformer's residual stream are packed into more directions than the stream has dimensions, so individual neurons or random directions are polysemantic — but a sparse, overcomplete dictionary trained to reconstruct the stream should recover the underlying monosemantic basis (`zotero://select/library/items/BISJXERV`). Bricken et al. 2023 turned that motivation into a working method on a one-layer transformer; Templeton et al. 2024 scaled the method to Claude 3 Sonnet and demonstrated that clamping individual features steers behaviour ("Golden Gate Claude") (`zotero://select/library/items/ZW2ZNFWZ`). Lieberum et al. 2024 then trained and released >400 JumpReLU SAEs across every layer of Gemma 2 — this is the open substrate we're using here (`zotero://select/library/items/FRHBADGA`).

The 2025 reckoning matters for how we read the output: SAEBench (Karvonen et al., ICML 2025) and Kantamneni et al. 2025 ("Are Sparse Autoencoders Useful?") showed that raw SAE features without downstream task validation should not be treated as ground truth — DeepMind's mech-interp team explicitly deprioritised fundamental SAE research in Mar 2025 as a result (see `vault/03-areas/research/mech-interp-overview.md` § State of the art § "Methodological reckoning"). What this notebook produces is a *hypothesis-generation* tool — top differentially-activating features per contrast prompt — not a causal claim. Phase 3 (attribution graphs) is where the causal story gets tested.

## Choice of layer

Default: **layer 20** of Gemma-2-2B (26 layers total — roughly the late residual stream, where Lindsey et al. 2025 §6 localises the refusal direction). Change `LAYER` below to sweep — early layers tend to surface tokeniser-level features, late layers surface task/concept-level features.

## Citations

- Toy Models of Superposition (the why): Elhage et al. 2022 — `zotero://select/library/items/BISJXERV`. See `vault/03-areas/research/superposition-and-polysemanticity.md` § Mechanism.
- Sparse-autoencoder dictionary learning recipe: Bricken et al. 2023 — `zotero://select/library/items/86AZ56WI`. See `vault/03-areas/research/superposition-and-polysemanticity.md` § Mechanism § SAE-based decomposition mechanism.
- Scaling to a deployed model + feature clamping: Templeton et al. 2024 — `zotero://select/library/items/ZW2ZNFWZ`. See `vault/03-areas/research/mech-interp-overview.md` § State of the art § Steering and feature-based control.
- JumpReLU SAEs (Gemma Scope's training recipe): Rajamanoharan et al. 2024 — `zotero://select/library/items/P4I47CF6`. See `vault/03-areas/research/superposition-and-polysemanticity.md` § Mechanism § SAE-based decomposition mechanism.
- The Gemma Scope release: Lieberum et al. 2024 — `zotero://select/library/items/FRHBADGA`. See `vault/03-areas/research/mech-interp-overview.md` § State of the art § Frontier-scale feature dictionaries.
- Refusal-direction prior art (the behavioural target this contrast pair gestures at): Lindsey et al. 2025 — `zotero://select/library/items/5XZVJMTF`. See `vault/03-areas/research/mech-interp-overview.md` § State of the art § Steering and feature-based control.
- The "are SAE features useful" reckoning: Karvonen et al. 2025 — `zotero://select/library/items/8WRK5HU8`. See `vault/03-areas/research/mech-interp-overview.md` § State of the art § Methodological reckoning.

## 1. Imports + paths

This cell assumes `01_residual_stream.ipynb` has already run and produced `../data/phase1_residual_stream.pt`. If not, run that notebook first — we re-use its prompts (and therefore its tokenisation) so the Phase-2 features map cleanly onto Phase-1 diagnostics.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import urllib.error
import urllib.request
import json

import mech_interp

DATA_DIR = Path('..') / 'data'
DATA_DIR.mkdir(exist_ok=True)

torch.set_grad_enabled(False)  # inference-only; SAE encode is a forward pass
sns.set_theme(context='notebook', style='white')

PHASE1_CHECKPOINT = DATA_DIR / 'phase1_residual_stream.pt'
assert PHASE1_CHECKPOINT.exists(), (
    f'expected {PHASE1_CHECKPOINT} from 01_residual_stream.ipynb; run that first'
)

## 2. Configuration

The SAE is selected by `(release, sae_id)`. Gemma Scope's residual-stream canonical release pins width 16k (the most-studied size) at every layer.

The Neuronpedia source ID for the same SAE is `<layer>-gemmascope-res-16k` under model `gemma-2-2b` — `NP_SOURCE_ID` below is constructed from `LAYER` so the two stay in sync.

In [ ]:
LAYER = 20  # 0..25 valid for Gemma-2-2B; mid-to-late by default
TOPK = 8   # how many features to surface per direction (refusal-leaning, compliance-leaning)

SAE_RELEASE = 'gemma-scope-2b-pt-res-canonical'
SAE_ID = f'layer_{LAYER}/width_16k/canonical'

# Neuronpedia hosts the matching feature dashboards under this name.
NP_MODEL_ID = 'gemma-2-2b'
NP_SOURCE_ID = f'{LAYER}-gemmascope-res-16k'

print(f'SAE      : {SAE_RELEASE} :: {SAE_ID}')
print(f'NP source: {NP_MODEL_ID} / {NP_SOURCE_ID}')

## 3. Load Gemma-2-2B + re-cache `resid_post` at `LAYER`

Phase 1 cached `hook_resid_pre` for the layer-by-layer ladder. Gemma Scope's residual-stream SAEs are trained on activations *after* each block writes back to the stream — i.e., `hook_resid_post`. We re-cache just one layer here (cheap; one forward pass per prompt) so the SAE inputs match the SAE's training distribution.

`mech_interp.get_model` is memoised so this won't redownload weights or re-instantiate the model if it's already resident from a prior cell or from `00_setup_smoke_test.ipynb` run in the same kernel.

In [ ]:
# Ref: A Mathematical Framework for Transformer Circuits — zotero://select/library/items/53EFBS8T
DEVICE, DTYPE = mech_interp.select_device_dtype()
model = mech_interp.get_model('gemma-2-2b', device=DEVICE, dtype=DTYPE)
n_layers = model.cfg.n_layers
d_model = model.cfg.d_model
assert 0 <= LAYER < n_layers, f'LAYER={LAYER} out of range for n_layers={n_layers}'
print(f'device={DEVICE} dtype={DTYPE} n_layers={n_layers} d_model={d_model}')

# Re-use Phase 1 prompts so the (layer, position) frame is preserved.
phase1 = torch.load(PHASE1_CHECKPOINT, weights_only=False)
P_COMPLIANCE = phase1['prompts']['compliance']
P_REFUSAL = phase1['prompts']['refusal']
TOKEN_LABELS = phase1['token_labels']

print('compliance:', P_COMPLIANCE)
print('refusal   :', P_REFUSAL)
print('aligned tokens:', TOKEN_LABELS)

In [ ]:
def cache_resid_post_at_layer(prompt: str, layer: int) -> torch.Tensor:
    """Return (seq_len, d_model) resid_post activations at the given layer, on CPU/fp32."""
    hook_name = f'blocks.{layer}.hook_resid_post'
    tokens = model.to_tokens(prompt)
    _, cache = model.run_with_cache(tokens, names_filter=lambda n: n == hook_name)
    return cache[hook_name].squeeze(0).to(torch.float32).cpu()

resid_compliance = cache_resid_post_at_layer(P_COMPLIANCE, LAYER)
resid_refusal = cache_resid_post_at_layer(P_REFUSAL, LAYER)

# Truncate to common length so per-position comparisons line up with Phase 1.
min_len = min(resid_compliance.shape[0], resid_refusal.shape[0], len(TOKEN_LABELS))
resid_compliance = resid_compliance[:min_len]
resid_refusal = resid_refusal[:min_len]
token_labels = TOKEN_LABELS[:min_len]

print(f'resid_compliance shape: {tuple(resid_compliance.shape)}')
print(f'resid_refusal    shape: {tuple(resid_refusal.shape)}')

## 4. Load the Gemma Scope SAE

`SAE.from_pretrained` downloads and instantiates the SAE on first call (the JumpReLU-style activation lives inside the SAE module, not the model). It returns `(sae, cfg_dict, sparsity)` — the second and third are metadata we display for the record.

If this cell fails with a `huggingface_hub` 401, you need to `uv run huggingface-cli login` first (Gemma Scope is gated, like Gemma-2-2B itself).

In [ ]:
# Ref: Gemma Scope — zotero://select/library/items/FRHBADGA
# Ref: JumpReLU SAEs — zotero://select/library/items/P4I47CF6
from sae_lens import SAE

sae, sae_cfg, sae_sparsity = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=DEVICE,
)
sae.eval()

d_sae = sae.cfg.d_sae
print(f'SAE      d_in={sae.cfg.d_in}  d_sae={d_sae}  expansion={d_sae / sae.cfg.d_in:.1f}×')
print(f'activation: {sae.cfg.activation_fn_str}  L0 (train avg, if reported): {sae_sparsity}')

## 5. Encode activations through the SAE

For each (position, prompt) we get a `d_sae`-dimensional vector of feature activations. Gemma Scope's canonical width-16k SAE is trained with JumpReLU so most entries will be exactly zero — that's the point.

We then compute the **differential score** per feature:

$$\Delta_f = \overline{|f_{\text{refusal}}|} - \overline{|f_{\text{compliance}}|},$$

averaged over token positions. The top-k features by positive $\Delta$ are "refusal-leaning"; the top-k by negative $\Delta$ are "compliance-leaning". This is a coarse signal — a feature that fires on the divergent-token positions only would dilute under position averaging — but it's the right first cut for surfacing candidates worth a Neuronpedia lookup.

In [ ]:
# Move activations to the SAE's device for the encode pass.
acts_compliance = resid_compliance.to(DEVICE).to(sae.dtype)
acts_refusal = resid_refusal.to(DEVICE).to(sae.dtype)

with torch.no_grad():
    feats_compliance = sae.encode(acts_compliance).to(torch.float32).cpu()  # (seq_len, d_sae)
    feats_refusal = sae.encode(acts_refusal).to(torch.float32).cpu()

print(f'feats_compliance shape: {tuple(feats_compliance.shape)}  '
      f'nonzero={(feats_compliance != 0).sum().item()} / {feats_compliance.numel()}')
print(f'feats_refusal    shape: {tuple(feats_refusal.shape)}  '
      f'nonzero={(feats_refusal != 0).sum().item()} / {feats_refusal.numel()}')

# Per-feature mean |activation| across positions, then differential.
mean_abs_compliance = feats_compliance.abs().mean(dim=0)  # (d_sae,)
mean_abs_refusal = feats_refusal.abs().mean(dim=0)
delta = mean_abs_refusal - mean_abs_compliance  # positive ⇒ refusal-leaning

## 6. Top-k feature picks per direction

Refusal-leaning = features whose mean activation magnitude is higher on the refusal prompt; compliance-leaning = the reverse. We surface `TOPK` in each direction. A feature appearing in both with near-zero $\Delta$ is a "shared scaffold" feature firing on the prompt structure rather than the contrast — those are filtered by the absolute-value sort.

In [ ]:
topk_refusal = torch.topk(delta, k=TOPK).indices.tolist()
topk_compliance = torch.topk(-delta, k=TOPK).indices.tolist()

def feature_row(idx: int) -> dict:
    return {
        'feature_idx': idx,
        'delta': delta[idx].item(),
        'mean_abs_compliance': mean_abs_compliance[idx].item(),
        'mean_abs_refusal': mean_abs_refusal[idx].item(),
        'firing_positions_compliance': (feats_compliance[:, idx] != 0).nonzero().flatten().tolist(),
        'firing_positions_refusal': (feats_refusal[:, idx] != 0).nonzero().flatten().tolist(),
    }

picks = {
    'refusal_leaning': [feature_row(i) for i in topk_refusal],
    'compliance_leaning': [feature_row(i) for i in topk_compliance],
}

print('--- refusal-leaning ---')
for row in picks['refusal_leaning']:
    print(f"  feat {row['feature_idx']:>5d}  Δ={row['delta']:+.4f}  "
          f"compliance_mean_abs={row['mean_abs_compliance']:.4f}  "
          f"refusal_mean_abs={row['mean_abs_refusal']:.4f}")
print('--- compliance-leaning ---')
for row in picks['compliance_leaning']:
    print(f"  feat {row['feature_idx']:>5d}  Δ={row['delta']:+.4f}  "
          f"compliance_mean_abs={row['mean_abs_compliance']:.4f}  "
          f"refusal_mean_abs={row['mean_abs_refusal']:.4f}")

## 7. Visualise where the top features fire

For each picked feature, show the (TOPK × seq_len) heatmap of activations across token positions, side-by-side for compliance vs refusal. Useful for spotting features that fire only on the divergent tokens versus features that fire across the whole prompt.

In [ ]:
def plot_feature_grid(feature_indices: list[int], title: str):
    if not feature_indices:
        return
    grid_compliance = feats_compliance[:, feature_indices].T.numpy()  # (k, seq_len)
    grid_refusal = feats_refusal[:, feature_indices].T.numpy()
    vmax = max(np.abs(grid_compliance).max(), np.abs(grid_refusal).max(), 1e-6)
    fig, axes = plt.subplots(1, 2, figsize=(16, max(3, 0.45 * len(feature_indices) + 2)),
                             sharey=True)
    xticklabels = [t.strip() or '·' for t in token_labels]
    yticklabels = [f'f{idx}' for idx in feature_indices]
    for ax, mat, sub in zip(axes, [grid_compliance, grid_refusal],
                            ['compliance', 'refusal']):
        sns.heatmap(mat, ax=ax, cmap='viridis', vmin=0, vmax=vmax,
                    xticklabels=xticklabels, yticklabels=yticklabels,
                    cbar_kws={'shrink': 0.6})
        ax.set_title(f'{title} — {sub}')
        ax.set_xlabel('token position')
    axes[0].set_ylabel('feature idx')
    plt.tight_layout()
    plt.show()

plot_feature_grid(topk_refusal, 'refusal-leaning features')
plot_feature_grid(topk_compliance, 'compliance-leaning features')

## 8. Pull Neuronpedia autointerp labels

Neuronpedia hosts feature dashboards (top-activating prompts + autointerp descriptions) for every Gemma Scope feature. We hit the public REST endpoint:

`https://www.neuronpedia.org/api/feature/<model_id>/<source_id>/<feature_idx>`

Network failures (offline, rate limit, 5xx) are caught so the rest of the notebook still runs — autointerp is *nice to have* but not load-bearing; the differential heatmap above is the substantive signal.

Heuristic from the [Scaling Monosemanticity](https://transformer-circuits.pub/2024/scaling-monosemanticity/) appendix: autointerp labels are noisy on Gemma Scope (the autointerp prompts were not tuned per layer), so treat the returned `description` as a *prior over what the feature might be*, not the answer. Confirm with the top-activating prompts list (`activations`) when one looks load-bearing.

In [ ]:
# Ref: Scaling Monosemanticity — zotero://select/library/items/ZW2ZNFWZ
# Ref: Gemma Scope — zotero://select/library/items/FRHBADGA
def fetch_np_feature(feature_idx: int, timeout: float = 8.0) -> dict | None:
    url = f'https://www.neuronpedia.org/api/feature/{NP_MODEL_ID}/{NP_SOURCE_ID}/{feature_idx}'
    req = urllib.request.Request(url, headers={'User-Agent': 'mech-interp-codebase/0.1'})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return json.loads(resp.read().decode())
    except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, OSError) as exc:
        print(f'  feat {feature_idx}: network error ({exc!r}) — falling back to stub')
        return None

def summarise_np_payload(payload: dict | None) -> str:
    if payload is None:
        return '(no Neuronpedia payload)'
    explanations = payload.get('explanations') or []
    if explanations:
        # Most-recent explanation first
        desc = explanations[0].get('description') or '(empty description)'
    else:
        desc = payload.get('description') or '(no description on Neuronpedia)'
    return desc.strip()

print('Fetching Neuronpedia autointerp for refusal-leaning picks...')
for row in picks['refusal_leaning']:
    payload = fetch_np_feature(row['feature_idx'])
    row['np_description'] = summarise_np_payload(payload)
    print(f"  feat {row['feature_idx']:>5d}: {row['np_description'][:120]}")

print()
print('Fetching Neuronpedia autointerp for compliance-leaning picks...')
for row in picks['compliance_leaning']:
    payload = fetch_np_feature(row['feature_idx'])
    row['np_description'] = summarise_np_payload(payload)
    print(f"  feat {row['feature_idx']:>5d}: {row['np_description'][:120]}")

## 9. Write a one-paragraph human gloss per top feature

For each of the **three most load-bearing** picks (your judgement: which features tell the cleanest story about how the residual stream represents the refusal vs compliance contrast?), paste a one-paragraph human gloss into `GLOSSES` below. The gloss should answer:

1. **What does this feature seem to fire on?** Token-level (e.g., specific words) or concept-level (e.g., "harm topic", "instructional preamble").
2. **Where does it fire on this contrast pair?** Divergent tokens only, scaffold tokens only, both? Cross-reference the heatmap in §7.
3. **Does the Neuronpedia autointerp from §8 agree?** Be specific about *disagreement*: that's where the most learning sits.
4. **Causal vs correlational?** This pass is correlational by construction. Note in one sentence the smallest experiment that would test causality (typical answer: clamp this feature to zero in `sae.W_dec` and re-run a forward pass — Phase 3 is the structured version of that).

The spec (`vault/01-projects/code/mech-interp-codebase.md` § Tasks § Phase 2 task 3) asks for at least three feature glosses — pick the three most informative across both directions.

In [ ]:
# Fill in below after inspecting §6–§8 output. Keep each gloss to a single paragraph.
GLOSSES: dict[int, str] = {
    # Example shape — replace once you've run the cells above:
    # 12345: """
    # Fires on tokens that introduce harm-relevant topics ('lethal', 'toxin').
    # On this contrast pair it activates exclusively on the refusal prompt's
    # divergent slot — visible in §7 row 1. Neuronpedia autointerp called it
    # 'words related to dangerous substances', which agrees with the firing
    # pattern. Correlational only; the minimal causal test is to clamp this
    # feature to zero and check whether the next-token distribution loses its
    # refusal-direction lean — Phase 3 will do this via attribution.
    # """
}

for idx, gloss in GLOSSES.items():
    print(f'--- feat {idx} ---')
    print(gloss.strip())
    print()

## 10. Checkpoint for Phase 3

Phase 3 (attribution graph) needs to know which features matter on this prompt so it can highlight them in the rendered graph. We save the picks + glosses + layer choice so `03_attribution.ipynb` can annotate the right nodes without re-running the SAE encode.

In [ ]:
payload = {
    'model_name': 'gemma-2-2b',
    'sae': {
        'release': SAE_RELEASE,
        'sae_id': SAE_ID,
        'layer': LAYER,
        'd_sae': d_sae,
        'd_in': sae.cfg.d_in,
        'activation_fn_str': sae.cfg.activation_fn_str,
        'np_source_id': NP_SOURCE_ID,
    },
    'prompts': {'compliance': P_COMPLIANCE, 'refusal': P_REFUSAL},
    'token_labels': token_labels,
    'features': {
        'feats_compliance': feats_compliance,  # (seq_len, d_sae) cpu fp32
        'feats_refusal': feats_refusal,
        'delta': delta,                        # (d_sae,) cpu fp32
        'mean_abs_compliance': mean_abs_compliance,
        'mean_abs_refusal': mean_abs_refusal,
    },
    'picks': picks,
    'glosses': GLOSSES,
}
out_path = DATA_DIR / 'phase2_sae_features.pt'
torch.save(payload, out_path)
print(f'wrote {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)')

## 11. Observations to log

Jot the answers below into `## Log` of `vault/01-projects/code/mech-interp-codebase.md` and into the Phase-2 findings file the orchestrator will append back into the linked research notes.

Prompts:

1. **How sparse?** What fraction of the `d_sae` features fired at least once on each prompt? Gemma Scope's reported L0 at this layer is in `sae_sparsity` — does the observed sparsity match?
2. **Did differential picks make prose sense?** Did the refusal-leaning features intuitively read as "harm-topic" features, or did they read as something else (e.g., tokeniser quirks, formatting features)?
3. **Did Neuronpedia autointerp agree?** Were the autointerp descriptions for your top picks recognisable as the same feature you saw firing in §7, or were they orthogonal?
4. **Open question for Phase 3.** Which of the top-3 picked features is the cleanest target for an attribution-graph annotation? The answer goes into `03_attribution.ipynb` §6 (annotate node).

## Open issues to surface to the user

- Gemma-2-2B is a base model. The "refusal-leaning" features here may surface "this is a harm-adjacent topic" features rather than "I should refuse" features (those live in the instruction-tuned variant). The literature's refusal-direction work (Lindsey et al. 2025) is on `-it`; we're sketching a base-model proxy. Note this gap when reading the autointerp output.
- Position-averaged differential dilutes single-position signal. If a feature fires only on the `'lethal toxin'` slot but is silent elsewhere, its $\Delta$ is small. A future iteration could swap mean for *max over divergent positions* — cheap to do in §5 by indexing into `feats_*` directly.
- Single-layer SAE only. Cross-layer transcoders (Anthropic 2024 → Mar 2025 attribution-graph methods, `zotero://select/library/items/ZQIF7Z59`) catch features that emerge across layers and disappear within a single layer — the Gemma Scope width-16k single-layer SAEs miss those by construction. Phase 3 uses the cross-layer transcoders via `circuit-tracer` which addresses this.
- Autointerp limitations. Per the 2025 reckoning (`vault/03-areas/research/mech-interp-overview.md` § Methodological reckoning), SAE-derived feature labels do not consistently beat simple probing baselines on downstream tasks. The autointerp labels are a starting point, not ground truth — fill `GLOSSES` based on what *you* see firing, not based on what Neuronpedia says.